# Les fonctions en Python

Dans ce notebook :

- Pourquoi écrire des fonctions (principe DRY)
- Définir une fonction, la documenter, la typer
- Les différentes formes d'arguments (positionnels, keyword, `*args`, `**kwargs`)
- Les fonctions anonymes (`lambda`)
- Scope, fonctions pures et effets de bord

## 1. Pourquoi écrire des fonctions ?

Les fonctions servent à **factoriser du code** qu'on utilise plusieurs fois. C'est le principe **DRY** (*Don't Repeat Yourself*) :

> *"Every piece of knowledge must have a single, unambiguous, authoritative representation within a system."*  
> — Andy Hunt & Dave Thomas, *The Pragmatic Programmer*

Quand on se retrouve à copier-coller du code en changeant juste une valeur, c'est le signal qu'il faut écrire une fonction.

### Exemple : convertir des températures

#### Mauvais : copier-coller du code

In [ ]:
temperature_lyon = 12
temperature_marseille = 35
temperature_paris = -7

temp_fahrenheit_lyon = temperature_lyon * 9 / 5 + 32
temp_fahrenheit_marseille = temperature_marseille * 9 / 5 + 32
temp_fahrenheit_paris = temperature_paris * 9 / 5 + 32

Problèmes : si on se trompe dans la formule, on doit corriger partout. Si on ajoute une ville, on doit copier une ligne de plus. C'est fragile.

#### Mieux : une fonction + une structure de données

In [ ]:
def celsius_to_fahrenheit(t_in_celsius):
    return t_in_celsius * 9 / 5 + 32


temp_in_celsius = {"lyon": 12, "marseille": 35, "paris": -7, "aubagne": 32}

temp_in_fahrenheit = {}
for ville, valeur in temp_in_celsius.items():
    temp_in_fahrenheit[ville] = celsius_to_fahrenheit(valeur)

temp_in_fahrenheit

La formule est écrite **une seule fois**. Pour ajouter une ville, il suffit d'ajouter une entrée dans le dictionnaire.

## 2. Définir une fonction

Une fonction prend **0 ou plus** arguments en entrée, et renvoie **0 ou plus** valeurs en sortie.

### Syntaxe de base

In [ ]:
def additionner(a, b):
    return a + b


additionner(3, 5)

### Fonctions sans retour explicite

Une fonction qui ne contient pas de `return` renvoie implicitement `None`.

In [ ]:
def coucou():
    print("coucou")


resultat = coucou()
print("La fonction a retourné :", resultat)
print("resultat is None ?", resultat is None)

**Question** : pouvez-vous donner un exemple de fonction qui ne retourne rien, mais qui fait quand même quelque chose d'utile ?

<details>
<summary>Réponse</summary>

Par exemple `print()`, ou une fonction qui écrit dans un fichier, ou qui affiche l'heure :
```python
import time

def afficher_heure():
    print(time.strftime("%H:%M:%S"))
```
Ces fonctions ne renvoient rien, mais produisent un **effet de bord** (on va en reparler plus loin).
</details>

## 3. Documenter une fonction : la docstring

Une **docstring** est une chaîne de caractères écrite juste après la signature de la fonction, entre triples guillemets. C'est la façon standard de documenter une fonction en Python.

### Écrire une docstring

In [ ]:
def ajouter_deux_nombres(a, b):
    """Renvoie la somme de deux nombres.

    Exemple : ajouter_deux_nombres(2, 3) == 5
    """
    return a + b

La docstring est accessible avec `?` dans Jupyter, ou avec `help()` :

In [ ]:
ajouter_deux_nombres?

### Conventions de docstring

La convention officielle est la [**PEP 257**](https://www.python.org/dev/peps/pep-0257/). En pratique, plusieurs styles coexistent :

- **NumPy style** : utilisé par NumPy, pandas, scikit-learn — très structuré
- **Google style** : populaire aussi, plus compact
- **reStructuredText** : le format "natif" de Python, utilisé par Sphinx

En 2026, avec les LLM, soigner sa docstring est **plus important que jamais** : le LLM lit la docstring pour comprendre ce que fait votre fonction et générer du code qui l'utilise correctement. Une bonne docstring = un meilleur assistant.

## 4. Type hints

Les **type hints** (annotations de types) permettent d'indiquer le type attendu des arguments et du retour d'une fonction. Introduits par la [PEP 484](https://peps.python.org/pep-0484/), ils sont devenus la norme en Python moderne.

### Écrire des type hints

In [ ]:
def ajouter_deux_nombres(a: int, b: int) -> int:
    """Renvoie la somme de deux nombres entiers."""
    return a + b


ajouter_deux_nombres(3, 5)

### Les types ne sont pas *forcés* par Python

**Point crucial** : Python n'utilise **pas** les type hints à l'exécution. Ils sont là pour :
- aider les développeurs à comprendre l'intention
- permettre à des outils (comme `mypy`, `pyright`, ou votre éditeur) de détecter des erreurs avant l'exécution
- aider les LLM à générer du code cohérent

Si on appelle la fonction avec de mauvais types, Python ne proteste pas — tant que l'opération `+` existe pour ces types, ça marche :

In [ ]:
# Python accepte, même si on a dit "int" dans la signature
ajouter_deux_nombres("bonjour ", "le monde")

In [ ]:
import numpy as np

# Fonctionne aussi avec des tableaux numpy — Python ne vérifie pas
ajouter_deux_nombres(np.array([1, 2, 3]), np.array([4, 5, 6]))

### Typage dynamique et duck typing

Le fait que Python ne vérifie pas les types à l'exécution s'appelle le **typage dynamique** : les types des variables sont déterminés au moment où le code s'exécute, pas à la lecture du code. C'est l'opposé de langages comme Java ou C++ où chaque variable a un type figé déclaré à l'avance.

Ce typage dynamique permet un style de programmation appelé **duck typing**, qui vient de l'expression :

> *"If it walks like a duck and quacks like a duck, it's a duck."*

**L'idée** : on ne demande pas "quel est le *type* de cet objet ?", mais "est-ce que cet objet *sait faire* ce dont j'ai besoin ?". Si oui, on l'utilise — peu importe son type formel.

Notre fonction `ajouter_deux_nombres` en est l'illustration parfaite :

In [ ]:
# La même fonction "addition" marche avec des types radicalement différents,
# tant que l'opérateur + est défini pour ces types.

print(ajouter_deux_nombres(3, 5))                                # int + int     → 8
print(ajouter_deux_nombres(3.5, 5.2))                            # float + float → 8.7
print(ajouter_deux_nombres("bonjour ", "le monde"))              # str + str     → concaténation
print(ajouter_deux_nombres([1, 2], [3, 4]))                      # list + list   → concaténation
print(ajouter_deux_nombres(np.array([1, 2, 3]), np.array([4, 5, 6])))  # array + array → addition élément par élément

Python n'a jamais vérifié que `a` et `b` étaient des `int`. Il a juste exécuté `a + b`, et chaque type a répondu à sa manière. Si on passait un objet qui ne sait pas faire `+`, là Python lèverait une erreur — **mais seulement à l'exécution, au moment où l'opération est tentée**.

**Conséquence importante** : le duck typing rend le code plus flexible et plus réutilisable, mais aussi plus fragile. Les erreurs de type ne sont détectées qu'à l'exécution, parfois loin de la cause. C'est pour ça que les **type hints** (qu'on a vus juste avant) sont devenus populaires : ils donnent une documentation et permettent aux outils de détecter les incohérences *avant* l'exécution.

### Types plus complexes

On peut typer des listes, dictionnaires, optionnels, etc. :

In [ ]:
def moyenne(nombres: list[float]) -> float:
    """Calcule la moyenne d'une liste de nombres."""
    return sum(nombres) / len(nombres)


def inverser_dict(d: dict[str, int]) -> dict[int, str]:
    """Inverse les clés et valeurs d'un dictionnaire."""
    return {v: k for k, v in d.items()}


moyenne([1.0, 2.0, 3.0, 4.0])

**Pour aller plus loin** :
- [Guide RealPython sur le type checking](https://realpython.com/python-type-checking/)
- [pydantic](https://docs.pydantic.dev/) : une librairie qui, elle, **force** les types à l'exécution (très utilisé pour valider des données d'API)

## 5. Arguments positionnels, keyword et par défaut

### Arguments positionnels vs arguments keyword

Quand on appelle une fonction, on peut passer ses arguments de deux manières :
- **Positionnel** : dans l'ordre de la définition
- **Keyword** (nommé) : avec le nom de l'argument

In [ ]:
def presenter(prenom, nom, age):
    print(f"{prenom} {nom}, {age} ans")


# Appel positionnel : les valeurs suivent l'ordre de définition
presenter("Ada", "Lovelace", 36)

# Appel avec keywords : on peut changer l'ordre
presenter(age=36, nom="Lovelace", prenom="Ada")

# Mixte : positionnel d'abord, puis keyword
presenter("Ada", nom="Lovelace", age=36)

### Valeurs par défaut

On peut donner une valeur par défaut à un argument, qui s'appliquera si on ne le passe pas :

In [ ]:
def saluer(nom, salutation="Bonjour"):
    print(f"{salutation}, {nom} !")


saluer("Alice")                       # utilise la valeur par défaut
saluer("Bob", salutation="Salut")     # surcharge la valeur par défaut
saluer("Chloé", "Hey")                # en positionnel, ça marche aussi

### Règle : les positionnels avant les keywords

Il y a **deux règles strictes** à connaître :

1. **Dans l'appel**, les arguments positionnels viennent toujours avant les keywords
2. **Dans la définition**, les arguments avec valeur par défaut viennent toujours après ceux sans valeur par défaut

Les deux cellules suivantes vont **volontairement échouer**. Avant de les exécuter, essayez de prédire l'erreur.

In [ ]:
# Règle 1 — dans l'appel : positionnel APRÈS keyword → erreur
presenter(prenom="Ada", "Lovelace", 36)

In [ ]:
# Règle 2 — dans la définition : argument sans défaut APRÈS un argument avec défaut → erreur
def afficher(salutation="Bonjour", nom):
    print(f"{salutation}, {nom}")

## 6. Unpacking et l'opérateur `*`

### Unpacking d'une séquence

L'unpacking permet d'assigner les éléments d'un itérable (liste, tuple) à plusieurs variables d'un coup :

In [ ]:
drapeau = ("bleu", "blanc", "rouge")

a, b, c = drapeau  # unpacking : a="bleu", b="blanc", c="rouge"
print(a, b, c)

Le nombre de variables doit correspondre au nombre d'éléments. La cellule suivante va **volontairement échouer** :

In [ ]:
a, b = drapeau  # 2 variables pour 3 éléments → erreur

### L'étoile `*` pour capturer le reste

On peut utiliser `*` devant une variable pour lui faire capturer "tout le reste" sous forme de liste :

In [ ]:
premiere, *autres = ("bleu", "blanc", "rouge", "noir", "vert")
print("premiere :", premiere)
print("autres :", autres)

### Unpacking dans un appel de fonction

L'étoile `*` fonctionne aussi **dans l'autre sens** : pour dérouler une liste dans les arguments d'une fonction.

In [ ]:
def afficher_trois(a, b, c):
    print(f"a={a}, b={b}, c={c}")


couleurs = ["bleu", "blanc", "rouge"]

# Sans unpacking : on passe UNE liste en UN argument → erreur
# afficher_trois(couleurs)  # TypeError

# Avec unpacking : les 3 éléments sont passés comme 3 arguments
afficher_trois(*couleurs)

Avec `**`, on fait la même chose mais avec un dictionnaire qui devient des **keyword arguments** :

In [ ]:
elements = {"a": "eau", "b": "feu", "c": "air"}
afficher_trois(**elements)  # équivaut à afficher_trois(a="eau", b="feu", c="air")

## 7. Arguments variables : `*args` et `**kwargs`

### `*args` — accepter un nombre variable d'arguments positionnels

Parfois on veut écrire une fonction qui accepte un nombre **quelconque** d'arguments. On utilise `*args` :

In [ ]:
def multiplier(*nombres):
    """Multiplie tous les nombres passés en argument."""
    resultat = 1
    for n in nombres:
        resultat *= n
    return resultat


print(multiplier(2, 3))           # 6
print(multiplier(2, 3, 4, 5))     # 120
print(multiplier(*[1, 2, 3, 4]))  # 24 — on peut unpacker une liste à l'appel

Le nom `args` est une convention ; on pourrait l'appeler autrement (`nombres`, `valeurs`…). C'est **l'étoile** qui fait la magie, pas le nom.

### `**kwargs` — accepter un nombre variable d'arguments nommés

Symétriquement, `**kwargs` capture tous les arguments **nommés** dans un dictionnaire :

In [ ]:
def afficher_recette(nom_recette, **ingredients):
    print(f"Recette : {nom_recette}")
    for ingredient, quantite in ingredients.items():
        print(f"  - {ingredient} : {quantite}")


afficher_recette(
    "Omelette",
    oeufs="3",
    beurre="20g",
    sel="une pincée",
)

### Combiner les deux

On peut mélanger arguments normaux + `*args` + `**kwargs`. L'ordre obligatoire est :

```
def f(arg_normal, arg_default="x", *args, **kwargs):
```

In [ ]:
def tout_afficher(obligatoire, optionnel="défaut", *args, **kwargs):
    print(f"obligatoire = {obligatoire}")
    print(f"optionnel   = {optionnel}")
    print(f"args        = {args}")
    print(f"kwargs      = {kwargs}")


tout_afficher("a", "b", "c", "d", extra1=1, extra2=2)

### Un exemple où on rencontre `**kwargs` au quotidien : matplotlib

La fonction `plt.plot()` de matplotlib accepte une dizaine d'options optionnelles via `**kwargs`. C'est pour ça qu'on peut l'appeler de plein de manières différentes :

In [ ]:
import matplotlib.pyplot as plt

plt.plot(
    [1, 2, 4],
    [4, 5, 6],
    color="red",
    linewidth=3,
    linestyle="--",
    marker="o",
);

## 8. Fonctions anonymes : `lambda`

Une **lambda** est une fonction écrite en une seule expression, sans lui donner de nom. Très utile quand on a besoin d'une petite fonction ponctuelle.

### Syntaxe

In [ ]:
# Équivalent de :
# def f(x, y):
#     return x + y
#
# En une ligne avec lambda :
(lambda x, y: x + y)(3, 5)

### Quand utiliser `lambda` ?

L'usage typique : passer une petite fonction **en argument** à une autre fonction. C'est omniprésent en pandas :

```python
# Appliquer une transformation rapide à une colonne
df["prix_ttc"] = df["prix_ht"].apply(lambda x: x * 1.2)

# Trier par une clé calculée
sorted(["Alice", "bob", "Chloé"], key=lambda s: s.lower())
```

### Anti-pattern : ne pas assigner une lambda à une variable

La PEP 8 recommande **de ne pas assigner une lambda à une variable**. Dans ce cas, utilisez un `def` classique :

In [ ]:
# ❌ Déconseillé : lambda assignée à une variable
ajouter = lambda x, y: x + y

# ✅ Préféré : def classique
def ajouter(x, y):
    return x + y

Pourquoi ? Parce qu'avec un `def`, le nom de la fonction apparaît dans les tracebacks (ce qui aide à déboguer), et la fonction peut être trouvée par son nom via introspection. Une lambda assignée se retrouve avec le nom générique `<lambda>` dans les erreurs.

## 9. Scope, fonctions pures et effets de bord

### Scope : où vit une variable

Le **scope** (portée), c'est la règle qui dit *où* Python va chercher une variable quand elle est utilisée. À l'intérieur d'une fonction, Python cherche d'abord une variable **locale** (définie dans la fonction), puis seulement ensuite une variable **globale** (définie au niveau du notebook/module).

In [ ]:
x = 10  # variable globale


def ma_fonction():
    x = 5  # variable LOCALE — une NOUVELLE variable qui masque la globale
    print("dans la fonction, x =", x)


ma_fonction()
print("hors de la fonction, x =", x)  # la globale n'a pas bougé

Pour vraiment modifier la variable globale depuis la fonction, il faut le dire explicitement avec `global` (mais c'est très rarement une bonne idée) :

In [ ]:
compteur = 0


def incrementer():
    global compteur  # on déclare qu'on veut modifier la globale
    compteur += 1


incrementer()
incrementer()
incrementer()
print("compteur =", compteur)

### Fonctions pures vs effets de bord

Une fonction est **pure** si :
- elle ne dépend **que** de ses arguments
- elle ne modifie **rien** en dehors d'elle-même (à part renvoyer une valeur)
- pour les mêmes entrées, elle renvoie toujours la même sortie

Tout ce que fait une fonction en plus de renvoyer une valeur s'appelle un **effet de bord** (*side effect*) : `print`, écrire dans un fichier, modifier une variable globale, modifier un argument, etc.

En pratique, on cherche à écrire un maximum de fonctions pures : elles sont plus faciles à comprendre, à tester, et à déboguer.

#### Exemple d'effet de bord sur un argument

Le piège le plus courant : une fonction qui **modifie silencieusement la liste** qu'on lui passe.

In [ ]:
def ajouter_zero(liste):
    """Renvoie la liste avec un 0 ajouté à la fin."""
    liste.append(0)  # ← modifie la liste de l'appelant !
    return liste


ma_liste = [1, 2, 3]
nouvelle_liste = ajouter_zero(ma_liste)

print("nouvelle_liste :", nouvelle_liste)
print("ma_liste       :", ma_liste)  # surprise : ma_liste a changé aussi

Une version **pure** de la même fonction créerait une nouvelle liste au lieu de modifier l'original :

In [ ]:
def ajouter_zero_pure(liste):
    """Renvoie une NOUVELLE liste avec un 0 ajouté à la fin."""
    return liste + [0]  # + crée une nouvelle liste, ne modifie pas l'original


ma_liste = [1, 2, 3]
nouvelle_liste = ajouter_zero_pure(ma_liste)

print("nouvelle_liste :", nouvelle_liste)
print("ma_liste       :", ma_liste)  # ma_liste est intact

### Les effets de bord sont partout, pas seulement dans vos fonctions

En réalité, les effets de bord ne concernent pas que les fonctions que vous écrivez. Ils se cachent dans plein d'endroits qui font qu'un même code peut se comporter différemment selon le contexte :

- **Les méthodes de classes** : quand vous appelez `mon_objet.ajouter(x)`, la méthode modifie souvent l'état de l'objet (`self`). C'est normal, c'est le principe des classes, mais c'est un effet de bord.
- **Les modifications d'API externes** : une API REST qui renvoie des champs différents d'un jour à l'autre → votre code qui marchait hier casse aujourd'hui.
- **Les changements de version de librairie** : une fonction pandas qui existait en v1 est renommée ou supprimée en v2 → votre code casse **sans qu'une seule ligne ait été modifiée**. C'est exactement pour ça qu'on fige les versions dans un **environnement virtuel** (voir le notebook sur [uv](0_1-Mettre_en_place_son_env_de_dev_python_pratique_uv.ipynb)).
- **Les variables d'environnement, fichiers de config** : la même fonction peut renvoyer des résultats différents selon le `.env` chargé.

Résumé en une phrase : **quand vous lisez du code (le vôtre ou celui d'un LLM), demandez-vous toujours : qu'est-ce que cette fonction change en plus de ce qu'elle renvoie ?**

## 10. Quiz de fin de notebook

Pour chaque question, essayez de répondre **avant** d'ouvrir la réponse. Certaines questions peuvent avoir plusieurs bonnes réponses.

### Question 1 — Duck typing

Regardez les deux appels suivants :

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Appel n°1 — avec des listes Python
plt.plot([1, 2, 3], [1, 5, 6])
plt.show()

# Appel n°2 — avec des tableaux numpy
plt.plot(np.array([1, 2, 3]), np.array([1, 5, 6]))
plt.show()

**Pourquoi ces deux appels à la même fonction, avec des types d'arguments différents, produisent-ils le même résultat ?**

- **A.** Parce que `plt.plot` détecte le type des arguments et a une branche de code spéciale pour chaque cas (listes, tuples, arrays numpy, etc.).
- **B.** Parce que numpy convertit automatiquement les listes en arrays avant l'appel.
- **C.** Parce que `plt.plot` fait du **duck typing** : elle n'a pas besoin d'un type précis, elle a juste besoin d'objets sur lesquels elle peut itérer et lire des valeurs numériques. Listes et arrays numpy savent tous les deux faire ça.
- **D.** Parce que les type hints de `plt.plot` acceptent à la fois `list` et `np.ndarray`.

<details>
<summary>Réponse</summary>

**La bonne réponse est C.**

`plt.plot` est écrite de manière à accepter tout ce qui "ressemble à une séquence de nombres" — peu lui importe le type exact. C'est la philosophie du duck typing : tant que l'objet sait faire ce dont la fonction a besoin (être itérable, contenir des nombres), ça marche.

- A est faux : il n'y a pas de branche `if isinstance(x, list) ... elif isinstance(x, np.ndarray)`. Ce serait fragile et impossible à maintenir.
- B est faux : numpy ne convertit rien automatiquement. C'est matplotlib qui sait travailler avec les deux.
- D est faux : les type hints ne *forcent* jamais rien en Python, ils ne jouent aucun rôle à l'exécution.
</details>

### Question 2 — Effet de bord

Que va afficher la dernière ligne ?

In [ ]:
def doubler_valeurs(valeurs):
    for i in range(len(valeurs)):
        valeurs[i] *= 2
    return valeurs


mes_nombres = [1, 2, 3]
nouveaux = doubler_valeurs(mes_nombres)

print("nouveaux    :", nouveaux)
print("mes_nombres :", mes_nombres)

- **A.** `nouveaux : [2, 4, 6]` et `mes_nombres : [1, 2, 3]` (la liste d'origine est intacte)
- **B.** `nouveaux : [2, 4, 6]` et `mes_nombres : [2, 4, 6]` (les deux listes sont identiques)
- **C.** `nouveaux : [1, 2, 3]` et `mes_nombres : [1, 2, 3]` (rien n'a été modifié)
- **D.** Une `TypeError` est levée parce qu'on ne peut pas modifier une liste passée en argument.

<details>
<summary>Réponse</summary>

**La bonne réponse est B.**

La fonction `doubler_valeurs` fait un **effet de bord** : elle modifie la liste reçue en argument avec `valeurs[i] *= 2`. Comme `mes_nombres` et `valeurs` désignent le **même objet** en mémoire, modifier l'un modifie l'autre.

De plus, `nouveaux = doubler_valeurs(mes_nombres)` ne fait que récupérer la référence vers cette même liste (puisque la fonction fait `return valeurs`). Résultat : `nouveaux`, `mes_nombres` et `valeurs` désignent **tous les trois** la même liste en mémoire.

Pour éviter ce piège, il faudrait soit créer une nouvelle liste (`return [v * 2 for v in valeurs]`), soit copier la liste en entrée (`valeurs = valeurs.copy()`).
</details>

### Question 3 — Scope

Que va afficher ce code ?

In [ ]:
message = "global"


def afficher():
    message = "local"
    print("dans la fonction :", message)


afficher()
print("hors de la fonction :", message)

- **A.** `dans la fonction : local` puis `hors de la fonction : global`
- **B.** `dans la fonction : local` puis `hors de la fonction : local`
- **C.** `dans la fonction : global` puis `hors de la fonction : global`
- **D.** Une erreur : on ne peut pas redéfinir une variable globale dans une fonction.

<details>
<summary>Réponse</summary>

**La bonne réponse est A.**

Dans la fonction `afficher()`, la ligne `message = "local"` **crée une nouvelle variable locale** qui s'appelle aussi `message`. Elle n'a aucun lien avec la variable globale du même nom. C'est la règle du scope local en Python : toute assignation dans une fonction crée par défaut une variable locale.

Pour modifier la variable globale depuis la fonction, il faudrait utiliser le mot-clé `global message` — mais c'est généralement une mauvaise idée.
</details>

### Question 4 — `*args` et `**kwargs`

On a la fonction suivante :

```python
def f(a, b=2, *args, **kwargs):
    print(a, b, args, kwargs)
```

Lequel de ces appels va afficher `1 2 (3, 4) {'nom': 'Alice'}` ?

- **A.** `f(1, 2, 3, 4, nom="Alice")`
- **B.** `f(1, b=2, 3, 4, nom="Alice")`
- **C.** `f(a=1, b=2, 3, 4, nom="Alice")`
- **D.** `f(1, 2, args=(3, 4), nom="Alice")`

<details>
<summary>Réponse</summary>

**La bonne réponse est A.**

- `1` est capturé par `a` (positionnel)
- `2` est capturé par `b` (positionnel, écrase la valeur par défaut)
- `3, 4` sont capturés par `*args` sous forme de tuple `(3, 4)`
- `nom="Alice"` est capturé par `**kwargs` sous forme de dict `{'nom': 'Alice'}`

**B et C sont faux** : on ne peut pas avoir un argument positionnel (`3`, `4`) **après** un keyword argument (`b=2`). C'est la règle de la section 5 : positionnels d'abord, keywords ensuite.

**D est faux** : `args` et `kwargs` ne sont **pas** des noms de paramètres qu'on passe à l'appel. Ce sont les noms des variables qui **reçoivent** les arguments positionnels et nommés dans la fonction. Passer `args=(3, 4)` serait interprété comme un keyword argument nommé `args`, et atterrirait dans `kwargs` !
</details>

### Question 5 — Type hints

On a cette fonction :

```python
def compter_mots(texte: str) -> int:
    return len(texte.split())
```

Que se passe-t-il si on l'appelle avec `compter_mots(["bonjour", "le", "monde"])` (une liste, pas une string) ?

- **A.** Python lève immédiatement une `TypeError` parce que le type hint dit `str`.
- **B.** Python accepte l'appel sans broncher, et renvoie le nombre d'éléments de la liste (3).
- **C.** Python accepte l'appel, mais lève une `AttributeError` au moment d'exécuter `texte.split()`, parce qu'une liste n'a pas de méthode `split()`.
- **D.** Python lève un `TypeError` au moment d'exécuter `len(...)`.

<details>
<summary>Réponse</summary>

**La bonne réponse est C.**

- A est faux : Python **ignore complètement** les type hints à l'exécution. Elles sont purement décoratives.
- B est faux : même si le résultat final serait 3 (le `len` d'une liste de 3 éléments), Python n'atteint pas ce point.
- D est faux : `len(...)` n'est jamais appelé, car l'erreur survient avant.
- C est vrai : Python exécute la fonction, essaie `texte.split()`, et **là** découvre qu'une liste n'a pas de méthode `.split()` → `AttributeError: 'list' object has no attribute 'split'`. C'est exactement le piège du duck typing : l'erreur se produit loin de la cause, au moment où on tente une opération que l'objet ne sait pas faire.
</details>